<a href="https://colab.research.google.com/github/detektor777/colab_list_image/blob/main/osediff_super_resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
show_log = False #@param {type:"boolean"}

import os, shutil, re, subprocess as sp
from IPython.utils import io
import contextlib

@contextlib.contextmanager
def optional_capture(capture):
    if capture:
        with io.capture_output() as captured:
            yield captured
    else:
        yield None

with optional_capture(not show_log):
    if os.path.isdir('/content/OSEDiff'):
        shutil.rmtree('/content/OSEDiff')

    os.system('git clone -q --depth 1 https://github.com/cswry/OSEDiff.git')
    os.chdir('OSEDiff')

    # The repository's requirements.txt pins versions of some old packages (e.g.,
    # PyYAML==5.4.1), which lack precompiled wheels for the current Python/setuptools
    # in Colab — pip tries to build them from source and fails with "Getting requirements
    # to build wheel did not run successfully" EVEN BEFORE it reaches diffusers/
    # transformers/peft/accelerate/basicsr. Fix: remove torch/torchvision/
    # torchaudio/xformers (we use what is already in Colab) and UNPIN the strict
    # versions for all other packages, leaving only names — pip will pick the
    # actual version with a precompiled wheel itself.
    with open('requirements.txt') as f:
        raw_lines = f.readlines()

    cleaned = []
    for line in raw_lines:
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('--extra-index-url'):
            continue
        if re.match(r'^(torch|torchvision|torchaudio|xformers)([=<>~!]|$)', line):
            continue
        pkg = re.split(r'[=<>~!\[]', line)[0].strip()
        if pkg:
            cleaned.append(pkg)

    with open('requirements_colab.txt', 'w') as f:
        f.write('\n'.join(cleaned) + '\n')

    print('requirements_colab.txt:')
    print(open('requirements_colab.txt').read())

    # --- pip install with exit code check ----------------------------------------
    install_result = os.system('pip install -q -r requirements_colab.txt')
    if install_result != 0:
        print('⚠️ pip install returned an error, trying to install packages one by one '
              '(so that one problematic package does not block the rest)...')
        with open('requirements_colab.txt') as f:
            pkgs = [p.strip() for p in f if p.strip()]
        for pkg in pkgs:
            r = os.system(f'pip install -q "{pkg}"')
            if r != 0:
                print(f'   ⚠️ failed to install: {pkg} (skipping, see output above)')

    # accelerate is not explicitly listed in requirements.txt (pulled as a dependency
    # by diffusers/peft), but is explicitly needed for inference — installing separately just in case.
    os.system('pip install -q accelerate')
    os.system('pip install -q huggingface_hub')

    # --- Patch torchao: version incompatibility with recent peft -----------------
    # Colab defaults to torchao 0.10.0, and the current peft installed alongside
    # requirements_colab.txt calls unet.add_adapter(...) (see osediff.py::load_ckpt)
    # going into peft/tuners/lora/torchao.py::dispatch_torchao and calling
    # is_torchao_available(), which explicitly requires torchao >= 0.16.0,
    # otherwise it throws an ImportError before the LoRA adapter is even applied.
    # Upgrading torchao separately (after the main installation to avoid pulling
    # torch version conflicts during requirements_colab.txt dependency resolution).
    torchao_upgrade_result = os.system('pip install -q -U "torchao>=0.16.0"')
    if torchao_upgrade_result != 0:
        print('⚠️ Failed to upgrade torchao to version >=0.16.0 — LoRA adapter '
              '(unet.add_adapter) at the "Run" step might fail with ImportError.')
    else:
        print('torchao upgraded to version >=0.16.0 (required for peft/dispatch_torchao).')

    # --- Patch basicsr: incompatibility with recent torchvision ------------------
    # basicsr/data/degradations.py does `from torchvision.transforms.functional_tensor
    # import rgb_to_grayscale` — this module was removed from torchvision starting version
    # 0.17 (the function moved to torchvision.transforms.functional), causing a simple
    # `import basicsr` to crash with ModuleNotFoundError before any use.
    # Patching the import to the actual path in all installed files where it occurs.
    find_result = sp.run(
        ['bash', '-c',
         "python -c \"import basicsr, os; print(os.path.dirname(basicsr.__file__))\" 2>/dev/null"],
        capture_output=True, text=True
    )
    basicsr_dir = find_result.stdout.strip()
    if not basicsr_dir:
        show_result = sp.run(['pip', 'show', 'basicsr'], capture_output=True, text=True).stdout
        for line in show_result.splitlines():
            if line.startswith('Location:'):
                site_packages = line.split(':', 1)[1].strip()
                basicsr_dir = os.path.join(site_packages, 'basicsr')
                break

    if basicsr_dir and os.path.isdir(basicsr_dir):
        patched_files = sp.run(
            ['grep', '-rl', 'functional_tensor', basicsr_dir],
            capture_output=True, text=True
        ).stdout.split()
        for fpath in patched_files:
            os.system(
                f"sed -i 's/from torchvision\\.transforms\\.functional_tensor import rgb_to_grayscale/"
                f"from torchvision.transforms.functional import rgb_to_grayscale/' {fpath}"
            )
        print('basicsr: patched files:', patched_files or '(none required)')
    else:
        print('⚠️ Failed to find basicsr directory for patching — please check the installation manually.')

    # --- Patch RAM: hardcoded local path to bert-base-uncased --------------------
    # ram/models/utils.py::init_tokenizer() in the original repo references
    # the author's absolute local path (something like
    # '/home/notebook/data/group/LowLevelLLM/LLM/bert-base-uncased'), which does not
    # exist in Colab or any other machine. This causes test_osediff.py to crash with
    # huggingface_hub.errors.HFValidationError / "Can't load tokenizer"
    # during RAM model initialization (see issue #7 in cswry/OSEDiff).
    # Replacing the hardcoded path with 'bert-base-uncased' so the tokenizer
    # is downloaded from Hugging Face Hub as intended.
    ram_utils_candidates = sp.run(
        ['grep', '-rl', 'bert-base-uncased', os.getcwd()],
        capture_output=True, text=True
    ).stdout.split()

    bert_path_pattern = re.compile(r"""(['"])(?:/[^'"]*)?bert-base-uncased\1""")
    ram_patched_files = []
    for fpath in ram_utils_candidates:
        with open(fpath, encoding='utf-8') as f:
            src = f.read()
        new_src = bert_path_pattern.sub(lambda m: f"{m.group(1)}bert-base-uncased{m.group(1)}", src)
        if new_src != src:
            with open(fpath, 'w', encoding='utf-8') as f:
                f.write(new_src)
            ram_patched_files.append(fpath)

    if ram_patched_files:
        print('RAM tokenizer: hardcoded path to bert-base-uncased replaced with HF repo id in files:', ram_patched_files)
    else:
        print('RAM tokenizer patch: no changes required (path is already correct or file not found).')

    # --- Patch RAM: unreliable additional_special_tokens_ids property ------------
    # ram/models/utils.py::init_tokenizer() does
    # `tokenizer.enc_token_id = tokenizer.additional_special_tokens_ids[0]`.
    # In current transformers versions, this property under certain conditions
    # throws an internal AttributeError, which Python interprets as "attribute missing",
    # and BertTokenizer gives a misleading message "BertTokenizer has no attribute
    # additional_special_tokens_ids" before the RAM model even loads.
    # The special token '[ENC]' is already added via add_special_tokens by this point,
    # so we just get its ID directly via convert_tokens_to_ids — this is more reliable
    # and independent of the transformers version.
    # Also removing local_files_only=True from from_pretrained so the tokenizer
    # can download itself from Hugging Face Hub if it's missing in the local cache.
    utils_path = os.path.join(os.getcwd(), 'ram', 'models', 'utils.py')
    if os.path.exists(utils_path):
        with open(utils_path, encoding='utf-8') as f:
            utils_src = f.read()

        old_enc_line = "tokenizer.enc_token_id = tokenizer.additional_special_tokens_ids[0]"
        new_enc_line = "tokenizer.enc_token_id = tokenizer.convert_tokens_to_ids('[ENC]')"
        replaced_enc = old_enc_line in utils_src
        if replaced_enc:
            utils_src = utils_src.replace(old_enc_line, new_enc_line)

        old_local_only = ", local_files_only=True)"
        replaced_local_only = old_local_only in utils_src
        if replaced_local_only:
            utils_src = utils_src.replace(old_local_only, ")")

        if replaced_enc or replaced_local_only:
            with open(utils_path, 'w', encoding='utf-8') as f:
                f.write(utils_src)
            print(f'ram/models/utils.py: init_tokenizer() patched '
                  f'(enc_token_id={replaced_enc}, local_files_only removed={replaced_local_only}).')
        else:
            print('⚠️ Did not find expected lines in ram/models/utils.py::init_tokenizer — perhaps the file changed, patch skipped.')
    else:
        print('⚠️ File ram/models/utils.py not found — skipping init_tokenizer patch.')

    # --- Patch RAM/BERT: transformers removed/moved several names, including -----
    # --- get_head_mask method (removed from PreTrainedModel/ModuleUtilsMixin) ----
    # ram/models/bert.py AND ram/models/bert_lora.py (adapted BERT code from
    # an old transformers version, two almost identical files) make the same
    # import `from transformers.modeling_utils import (PreTrainedModel,
    # apply_chunking_to_forward, find_pruneable_heads_and_indices, prune_linear_layer)`.
    # In current transformers, apply_chunking_to_forward and prune_linear_layer
    # moved to transformers.pytorch_utils, and find_pruneable_heads_and_indices
    # was removed entirely (used only for manual BERT attention head pruning,
    # which RAM inference in OSEDiff does not do). In addition, the get_head_mask
    # method (and its helper _convert_head_mask_to_5d) was removed from PreTrainedModel,
    # which BertModel.forward() calls directly as self.get_head_mask(...) —
    # causing AttributeError: 'BertModel' object has no attribute 'get_head_mask'.
    # Patching the import to a fault-tolerant version in ALL repo files where it occurs,
    # and simultaneously monkeypatching both methods back onto the PreTrainedModel
    # class if they are missing — this fixes it once for all models using this class.
    old_import = (
        "from transformers.modeling_utils import (\n"
        "    PreTrainedModel,\n"
        "    apply_chunking_to_forward,\n"
        "    find_pruneable_heads_and_indices,\n"
        "    prune_linear_layer,\n"
        ")"
    )
    new_import = (
        "import transformers.modeling_utils as _mod_transformers_modeling_utils\n"
        "try:\n"
        "    import transformers.pytorch_utils as _mod_transformers_pytorch_utils\n"
        "except ModuleNotFoundError:\n"
        "    _mod_transformers_pytorch_utils = None\n"
        "def _resolve_transformers_symbol(name):\n"
        "    for _mod in (_mod_transformers_modeling_utils, _mod_transformers_pytorch_utils):\n"
        "        if _mod is not None and hasattr(_mod, name):\n"
        "            return getattr(_mod, name)\n"
        "    return type(name, (object,), {})\n"
        "PreTrainedModel = _resolve_transformers_symbol('PreTrainedModel')\n"
        "apply_chunking_to_forward = _resolve_transformers_symbol('apply_chunking_to_forward')\n"
        "find_pruneable_heads_and_indices = _resolve_transformers_symbol('find_pruneable_heads_and_indices')\n"
        "prune_linear_layer = _resolve_transformers_symbol('prune_linear_layer')\n"
        "if not hasattr(PreTrainedModel, 'get_head_mask'):\n"
        "    def _convert_head_mask_to_5d(self, head_mask, num_hidden_layers):\n"
        "        if head_mask.dim() == 1:\n"
        "            head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)\n"
        "            head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)\n"
        "        elif head_mask.dim() == 2:\n"
        "            head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)\n"
        "        head_mask = head_mask.to(dtype=self.dtype)\n"
        "        return head_mask\n"
        "    def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):\n"
        "        if head_mask is not None:\n"
        "            head_mask = self._convert_head_mask_to_5d(head_mask, num_hidden_layers)\n"
        "            if is_attention_chunked is True:\n"
        "                head_mask = head_mask.unsqueeze(-1)\n"
        "        else:\n"
        "            head_mask = [None] * num_hidden_layers\n"
        "        return head_mask\n"
        "    PreTrainedModel._convert_head_mask_to_5d = _convert_head_mask_to_5d\n"
        "    PreTrainedModel.get_head_mask = _get_head_mask"
    )

    bert_patched_files = []
    for root, dirs, files in os.walk(os.getcwd()):
        if '.git' in dirs:
            dirs.remove('.git')
        for fname in files:
            if not fname.endswith('.py'):
                continue
            fpath = os.path.join(root, fname)
            with open(fpath, encoding='utf-8') as f:
                file_src = f.read()
            if old_import in file_src:
                file_src = file_src.replace(old_import, new_import)
                with open(fpath, 'w', encoding='utf-8') as f:
                    f.write(file_src)
                bert_patched_files.append(os.path.relpath(fpath, os.getcwd()))

    if bert_patched_files:
        print('Import from transformers.modeling_utils made fault-tolerant (+ get_head_mask) in files:', bert_patched_files)
    else:
        print('⚠️ Did not find the expected transformers.modeling_utils import block in any file — perhaps the repo structure changed.')

    # --- Patch RAM/BERT: init_weights() is deprecated, post_init() is required -
    # ram/models/bert.py and ram/models/bert_lora.py at the end of __init__
    # for each model class (BertModel, BertLMHeadModel, etc.) call self.init_weights() —
    # this is how it was done in older versions of transformers. In current versions,
    # init_weights() expects self.all_tied_weights_keys to be precalculated, but
    # it is only calculated by post_init() (which itself calls init_weights() at the end).
    # Calling init_weights() directly bypassing post_init() causes
    # AttributeError: 'BertModel' object has no attribute 'all_tied_weights_keys'
    # inside tie_weights(). Changing self.init_weights() to self.post_init()
    # in all repo files where such a call occurs.
    init_weights_patched_files = []
    for root, dirs, files in os.walk(os.getcwd()):
        if '.git' in dirs:
            dirs.remove('.git')
        for fname in files:
            if not fname.endswith('.py'):
                continue
            fpath = os.path.join(root, fname)
            with open(fpath, encoding='utf-8') as f:
                file_src = f.read()
            new_file_src = re.sub(
                r'(?<!def )(?<!\.)\bself\.init_weights\(\)',
                'self.post_init()',
                file_src,
            )
            if new_file_src != file_src:
                with open(fpath, 'w', encoding='utf-8') as f:
                    f.write(new_file_src)
                init_weights_patched_files.append(os.path.relpath(fpath, os.getcwd()))

    if init_weights_patched_files:
        print('self.init_weights() replaced with self.post_init() in files:', init_weights_patched_files)
    else:
        print('⚠️ Did not find self.init_weights() calls in any file — perhaps the repo structure changed.')

    # --- Patch osediff.py: scheduler.timesteps might not match hardcoded 999 ---
    # In all classes (OSEDiff_test, etc.) the code does:
    #   self.noise_scheduler.set_timesteps(1, device="cuda")
    #   ...
    #   self.timesteps = torch.tensor([999], device="cuda").long()
    # and then calls self.noise_scheduler.step(model_pred, self.timesteps, ...),
    # which internally (DDPMScheduler.previous_timestep) searches for the index
    # of the value 999 in ITS OWN internal list self.noise_scheduler.timesteps,
    # calculated via set_timesteps(1, ...). In current versions of diffusers for
    # num_inference_steps=1, this list depending on timestep_spacing in the scheduler config
    # is not necessarily equal to [999] (it could be [0], for example) — then 999
    # is not found and it crashes with IndexError: index 0 is out of bounds for dimension
    # 0 with size 0. Fixed by forcibly overwriting self.noise_scheduler.timesteps
    # to [999] right after set_timesteps(1, ...), so it matches the model's hardcode.
    scheduler_timesteps_patched_files = []
    for root, dirs, files in os.walk(os.getcwd()):
        if '.git' in dirs:
            dirs.remove('.git')
        for fname in files:
            if not fname.endswith('.py'):
                continue
            fpath = os.path.join(root, fname)
            with open(fpath, encoding='utf-8') as f:
                file_src = f.read()

            old_set_timesteps_line = 'self.noise_scheduler.set_timesteps(1, device="cuda")'
            new_set_timesteps_block = (
                'self.noise_scheduler.set_timesteps(1, device="cuda")\n'
                '        self.noise_scheduler.timesteps = torch.tensor([999], device="cuda").long()'
            )
            if old_set_timesteps_line in file_src:
                file_src = file_src.replace(old_set_timesteps_line, new_set_timesteps_block)
                with open(fpath, 'w', encoding='utf-8') as f:
                    f.write(file_src)
                scheduler_timesteps_patched_files.append(os.path.relpath(fpath, os.getcwd()))

    if scheduler_timesteps_patched_files:
        print('noise_scheduler.timesteps forcibly set to [999] in files:', scheduler_timesteps_patched_files)
    else:
        print('⚠️ Did not find set_timesteps(1, device="cuda") calls in any file — perhaps the repo structure changed.')

    # --- Patch UNet2DConditionModel: add_adapter moved to PeftAdapterMixin -----
    # Local copy of models/unet_2d_condition.py is a diffusers fork for an older version
    # of the library, when LoRA/PEFT methods (add_adapter/delete_adapter, etc.) were
    # part of UNet2DConditionLoadersMixin. In current diffusers, they were moved to
    # a separate PeftAdapterMixin, and the base classes here are declared without it —
    # causing self.unet.add_adapter(...) to crash with AttributeError. Adding
    # PeftAdapterMixin to the base classes list (via getattr with a fallback stub
    # in case it is also missing in some diffusers version).
    unet_cond_path = os.path.join(os.getcwd(), 'models', 'unet_2d_condition.py')
    if os.path.exists(unet_cond_path):
        with open(unet_cond_path, encoding='utf-8') as f:
            unet_src = f.read()

        old_class_decl = 'class UNet2DConditionModel(ModelMixin, ConfigMixin, UNet2DConditionLoadersMixin):'
        if old_class_decl in unet_src:
            peft_mixin_shim = (
                "import diffusers.loaders as _mod_diffusers_loaders_for_peft\n"
                "PeftAdapterMixin = getattr(_mod_diffusers_loaders_for_peft, 'PeftAdapterMixin', "
                "type('PeftAdapterMixin', (object,), {}))\n\n\n"
            )
            new_class_decl = (
                'class UNet2DConditionModel(ModelMixin, ConfigMixin, UNet2DConditionLoadersMixin, PeftAdapterMixin):'
            )
            unet_src = unet_src.replace(old_class_decl, peft_mixin_shim + new_class_decl)
            with open(unet_cond_path, 'w', encoding='utf-8') as f:
                f.write(unet_src)
            print('models/unet_2d_condition.py: added PeftAdapterMixin to UNet2DConditionModel base classes (needed for add_adapter).')
        else:
            print('⚠️ Did not find expected UNet2DConditionModel class declaration — perhaps the file changed, patch skipped.')
    else:
        print('⚠️ File models/unet_2d_condition.py not found — skipping PeftAdapterMixin patch.')

    # --- Patch AutoencoderKL: add_adapter also moved to PeftAdapterMixin -------
    # Similar to UNet2DConditionModel, the local copy of models/autoencoder_kl.py is
    # a diffusers fork for an older version, where LoRA/PEFT methods (add_adapter, etc.)
    # were available without a separate mixin. In current diffusers, they were moved
    # to PeftAdapterMixin, and the AutoencoderKL base classes here are declared without it —
    # causing self.vae.add_adapter(...) in osediff.py::load_ckpt to crash with
    # AttributeError: 'AutoencoderKL' object has no attribute 'add_adapter'.
    vae_path = os.path.join(os.getcwd(), 'models', 'autoencoder_kl.py')
    if os.path.exists(vae_path):
        with open(vae_path, encoding='utf-8') as f:
            vae_src = f.read()

        old_vae_class_decl = 'class AutoencoderKL(ModelMixin, ConfigMixin, FromOriginalVAEMixin):'
        if old_vae_class_decl in vae_src:
            vae_peft_mixin_shim = (
                "import diffusers.loaders as _mod_diffusers_loaders_for_peft_vae\n"
                "PeftAdapterMixin = getattr(_mod_diffusers_loaders_for_peft_vae, 'PeftAdapterMixin', "
                "type('PeftAdapterMixin', (object,), {}))\n\n\n"
            )
            new_vae_class_decl = (
                'class AutoencoderKL(ModelMixin, ConfigMixin, FromOriginalVAEMixin, PeftAdapterMixin):'
            )
            vae_src = vae_src.replace(old_vae_class_decl, vae_peft_mixin_shim + new_vae_class_decl)
            with open(vae_path, 'w', encoding='utf-8') as f:
                f.write(vae_src)
            print('models/autoencoder_kl.py: added PeftAdapterMixin to AutoencoderKL base classes (needed for add_adapter).')
        else:
            print('⚠️ Did not find expected AutoencoderKL class declaration — perhaps the file changed, patch skipped.')
    else:
        print('⚠️ File models/autoencoder_kl.py not found — skipping PeftAdapterMixin patch.')

    # --- Universal patch: OSEDiff was written for an older version of diffusers -
    # In the installed Colab diffusers version, some classes (loading mixins,
    # individual embedding modules like PositionNet, etc.) have been renamed or
    # removed, and some submodule paths (e.g. diffusers.models.unet_2d_blocks)
    # ENTIRELY moved to the diffusers.models.unets.* subpackage. Instead of
    # fixing one ImportError after another precisely, we scan ALL .py files
    # in the repo and rewrite EVERY import like `from diffusers... import (A, B, C)`
    # (including multi-line ones, in parentheses) to a safe version: first try
    # to import the module by its original path, and if it no longer exists
    # (ModuleNotFoundError) — try the current diffusers.models.unets.<rest> path.
    # Then, for each name, use getattr(module, 'Name', fallback_stub). If the
    # symbol actually exists in the installed diffusers version — it is used
    # (no functionality loss), if removed/renamed — an empty stub class is
    # substituted instead of crashing on import.
    import_pattern = re.compile(
        r'^(?P<indent>[ \t]*)from (?P<module>diffusers[\w.]*) import '
        r'(?P<body>\([^()]*\)|[^\n]+)$',
        re.MULTILINE
    )

    def _parse_names(body):
        body = body.strip()
        if body.startswith('(') and body.endswith(')'):
            body = body[1:-1]
        names = []
        for part in body.split(','):
            part = part.split('#')[0].strip()
            if not part:
                continue
            if ' as ' in part:
                orig, alias = [p.strip() for p in part.split(' as ', 1)]
                names.append((orig, alias))
            else:
                names.append((part, part))
        return names

    repo_root = os.getcwd()
    patched_summary = []
    for root, dirs, files in os.walk(repo_root):
        if '.git' in dirs:
            dirs.remove('.git')
        for fname in files:
            if not fname.endswith('.py'):
                continue
            fpath = os.path.join(root, fname)
            with open(fpath, encoding='utf-8') as f:
                src = f.read()

            matches = list(import_pattern.finditer(src))
            if not matches:
                continue

            new_src = src
            offset = 0
            touched_names = []
            for m in matches:
                module = m.group('module')
                names = _parse_names(m.group('body'))
                if not names:
                    continue
                mod_alias = '_mod_' + re.sub(r'\W', '_', module)

                fallback_module = None
                if module.startswith('diffusers.models.') and '.unets.' not in module:
                    fallback_module = module.replace('diffusers.models.', 'diffusers.models.unets.', 1)

                if fallback_module:
                    lines = [
                        f'{m.group("indent")}try:',
                        f'{m.group("indent")}    import {module} as {mod_alias}',
                        f'{m.group("indent")}except ModuleNotFoundError:',
                        f'{m.group("indent")}    import {fallback_module} as {mod_alias}',
                    ]
                else:
                    lines = [f'{m.group("indent")}import {module} as {mod_alias}']

                for orig, alias in names:
                    lines.append(
                        f"{m.group('indent')}{alias} = getattr({mod_alias}, '{orig}', "
                        f"type('{orig}', (object,), {{}}))"
                    )
                    touched_names.append(orig)
                replacement = '\n'.join(lines)

                start, end = m.start() + offset, m.end() + offset
                new_src = new_src[:start] + replacement + new_src[end:]
                offset += len(replacement) - (m.end() - m.start())

            if new_src != src:
                with open(fpath, 'w', encoding='utf-8') as f:
                    f.write(new_src)
                rel_path = os.path.relpath(fpath, repo_root)
                patched_summary.append(f'{rel_path}: {touched_names}')

    if patched_summary:
        print('diffusers imports patched to a safe (getattr) version:')
        for line in patched_summary:
            print('  -', line)
    else:
        print('diffusers imports patch: no changes required.')

    # --- Check that critical packages can actually be imported -----------------
    required_modules = ['diffusers', 'transformers', 'peft', 'accelerate', 'basicsr', 'einops']
    missing = []
    for module_name in required_modules:
        try:
            __import__(module_name)
        except ImportError as e:
            missing.append(f'{module_name} ({e})')

    if missing:
        raise RuntimeError(
            f'Critical modules failed to install/import: {missing}. '
            f'If "show_log" is off, please enable it, re-run the cell, and check the output for errors.'
        )
    else:
        print('✅ All critical packages are installed and importable:', required_modules)

    # --- Final check: the osediff package itself and both RAM versions must be importable ---
    # Checking not only osediff, but also ram.models.ram (uses bert.py) and
    # ram.models.ram_lora (uses bert_lora.py) — this is where transformers imports
    # used to crash, while a simple `import osediff` wouldn't touch them.
    check = sp.run(
        ['python', '-c',
         'import sys; sys.path.insert(0, "."); import osediff; import ram.models.ram; import ram.models.ram_lora; print("osediff OK")'],
        capture_output=True, text=True
    )
    print(check.stdout.strip())
    if check.returncode != 0:
        print('⚠️ The osediff/ram module still cannot be imported. Full traceback:')
        print(check.stderr)
        raise RuntimeError(
            'import osediff/ram failed with an error — see the traceback above '
            '(enable "show_log" if it is disabled) and provide its text.'
        )

    import torch
    print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

    # --- Weights -----------------------------------------------------------------
    from huggingface_hub import hf_hub_download, snapshot_download

    os.makedirs('preset/models', exist_ok=True)

    def file_ok(path, min_bytes=1024):
        return os.path.exists(path) and os.path.getsize(path) >= min_bytes

    sd21_dir = 'preset/models/stable-diffusion-2-1-base'
    if not os.path.isdir(sd21_dir) or not os.path.exists(os.path.join(sd21_dir, 'model_index.json')):
        print('Downloading SD21 Base (Manojb/stable-diffusion-2-1-base)...')
        snapshot_download(
            repo_id='Manojb/stable-diffusion-2-1-base',
            local_dir=sd21_dir,
            allow_patterns=[
                'tokenizer/*', 'text_encoder/*', 'vae/*', 'unet/*', 'scheduler/*',
                'model_index.json',
            ],
        )
        print('Done:', sd21_dir)
    else:
        print('SD21 Base is already present, skipping download:', sd21_dir)

    ram_path = 'preset/models/ram_swin_large_14m.pth'
    if not file_ok(ram_path, min_bytes=10_000_000):
        print('Downloading RAM (xinyu1205/recognize-anything, HF Space)...')
        downloaded = hf_hub_download(
            repo_id='xinyu1205/recognize-anything',
            repo_type='space',
            filename='ram_swin_large_14m.pth',
        )
        shutil.copy(downloaded, ram_path)
        print('Done:', ram_path)
    else:
        print('RAM is already present, skipping download:', ram_path)

    dape_path = 'preset/models/DAPE.pth'
    if not file_ok(dape_path, min_bytes=1_000_000):
        print('Downloading DAPE.pth (Guaishou74851/AdcSR mirror)...')
        downloaded = hf_hub_download(
            repo_id='Guaishou74851/AdcSR',
            filename='weight/pretrained/DAPE.pth',
        )
        shutil.copy(downloaded, dape_path)
        print('Done:', dape_path)
    else:
        print('DAPE is already present, skipping download:', dape_path)

    osediff_path = 'preset/models/osediff.pkl'
    if not file_ok(osediff_path, min_bytes=1_000_000):
        print('Downloading osediff.pkl (Guaishou74851/AdcSR mirror)...')
        downloaded = hf_hub_download(
            repo_id='Guaishou74851/AdcSR',
            filename='weight/pretrained/osediff.pkl',
        )
        shutil.copy(downloaded, osediff_path)
        print('Done:', osediff_path)
    else:
        print('osediff.pkl is already present, skipping download:', osediff_path)

    print('OSEDiff cloned, dependencies installed, weights are in place.')

In [ ]:
#@title ##**Upload Images** { display-mode: "form" }
import os
import shutil
from google.colab import files

upload_folder = './inputs/user_upload'
result_folder = './results/user_upload'

if os.path.isdir(upload_folder):
    shutil.rmtree(upload_folder)
if os.path.isdir(result_folder):
    shutil.rmtree(result_folder)

os.makedirs(upload_folder, exist_ok=True)
os.makedirs(result_folder, exist_ok=True)

# Upload images — you can select multiple files at once
uploaded = files.upload()
for filename in uploaded.keys():
    dst_path = os.path.join(upload_folder, filename)
    print(f'Moved {filename} -> {dst_path}')
    shutil.move(filename, dst_path)

print(f'\n{len(uploaded)} image(s) uploaded to {upload_folder}')


In [ ]:
#@title ##**Run** { display-mode: "form" }
import os, re, gc, codecs, torch, psutil, subprocess, sys, time, threading, traceback, shutil

start_time_main = time.time()

if os.path.basename(os.getcwd()) != 'OSEDiff':
    if os.path.isdir('/content/OSEDiff'):
        os.chdir('/content/OSEDiff')
        print(f'⚠️ Working directory was reset. Switched to {os.getcwd()}')
    else:
        raise RuntimeError('Folder /content/OSEDiff not found. Please run the "Install" cell.')

if 'upload_folder' not in globals():
    upload_folder = './inputs/user_upload'
    print(f'⚠️ upload_folder was not defined. Using: {upload_folder}')
if 'result_folder' not in globals():
    result_folder = './results/user_upload'
    print(f'⚠️ result_folder was not defined. Using: {result_folder}')

if not os.path.isdir(upload_folder) or not os.listdir(upload_folder):
    raise RuntimeError(f'No images found in {upload_folder}. Please run "Upload Images".')
os.makedirs(result_folder, exist_ok=True)

# --- Check required weights before running ----------------------------------
required_paths = {
    'SD21 Base': 'preset/models/stable-diffusion-2-1-base/model_index.json',
    'RAM': 'preset/models/ram_swin_large_14m.pth',
    'DAPE': 'preset/models/DAPE.pth',
    'osediff.pkl': 'preset/models/osediff.pkl',
}
missing_weights = [name for name, p in required_paths.items() if not os.path.exists(p)]
if missing_weights:
    raise RuntimeError(f'Missing weights: {missing_weights}. Please re-run the "Install" cell.')

# --- Inspect input files ------------------------------------------------------
from PIL import Image, UnidentifiedImageError
Image.MAX_IMAGE_PIXELS = None  # disable the "decompression bomb" guard for large images

print('--- Checking input files ---')
valid_input_files = []
for fname in sorted(os.listdir(upload_folder)):
    fpath = os.path.join(upload_folder, fname)
    if not os.path.isfile(fpath):
        continue
    size_mb = os.path.getsize(fpath) / 1024**2
    try:
        with Image.open(fpath) as im:
            im.load()
            w, h = im.size
            print(f'  ✅ {fname}: {w}x{h} ({w*h/1e6:.1f} MP), format={im.format}, mode={im.mode}, {size_mb:.1f} MB')
            valid_input_files.append(fname)
    except UnidentifiedImageError:
        print(f'  ❌ {fname}: PIL could not identify this file as an image ({size_mb:.1f} MB)')
    except Exception as e:
        print(f'  ❌ {fname}: error while opening ({size_mb:.1f} MB) -> {type(e).__name__}: {e}')
        traceback.print_exc()

if not valid_input_files:
    raise RuntimeError('None of the files in upload_folder could be opened as an image. See the log above.')
print(f'--- Valid images found: {len(valid_input_files)} ---\n')

# --- IMPORTANT: test_osediff.py only looks for files matching *.png --------
# Source line: image_names = sorted(glob.glob(f'{args.input_image}/*.png'))
# So any .jpg/.jpeg/.bmp/.webp file is silently skipped even though PIL can
# open it fine. We convert non-png files to .png so the script can find them.
print('--- Converting input files to .png (required by test_osediff.py) ---')
converted_count = 0
for fname in valid_input_files:
    base, ext = os.path.splitext(fname)
    if ext.lower() == '.png':
        continue  # already png, no conversion needed
    src_path = os.path.join(upload_folder, fname)
    dst_path = os.path.join(upload_folder, base + '.png')
    if os.path.exists(dst_path):
        print(f'  ⏭️ {fname}: {base}.png already exists, skipping conversion')
        continue
    with Image.open(src_path) as im:
        im.convert('RGB').save(dst_path, format='PNG')
    print(f'  🔄 {fname} -> {base}.png (converted)')
    converted_count += 1

png_files_now = [f for f in os.listdir(upload_folder) if f.lower().endswith('.png')]
print(f'--- .png files in upload_folder now: {len(png_files_now)} (converted this run: {converted_count}) ---\n')

if not png_files_now:
    raise RuntimeError('No .png files found after conversion — test_osediff.py will find nothing to process.')

# --- Colab interface settings ------------------------------------------------
task = "sr" #@param ["sr", "face"]
upscale = 3 #@param {type:"slider", min:1, max:8, step:1}
align_method = "adain" #@param ["adain", "wavelet", "nofix"]
mixed_precision = "fp16" #@param ["fp16", "fp32"]
merge_and_unload_lora = True #@param {type:"boolean"}
low_vram_tiled_sampling = True #@param {type:"boolean"}
vae_encoder_tiled_size = 512 #@param {type:"integer"}
vae_decoder_tiled_size = 128 #@param {type:"integer"}
latent_tiled_size = 96 #@param {type:"integer"}
latent_tiled_overlap = 16 #@param {type:"integer"}
show_log = False #@param {type:"boolean"}

osediff_path = 'preset/models/osediff_face.pkl' if task == 'face' else 'preset/models/osediff.pkl'
if task == 'face' and not os.path.exists(osediff_path):
    print(f'⚠️ {osediff_path} not found. Falling back to osediff.pkl (standard SR).')
    osediff_path = 'preset/models/osediff.pkl'
if task == 'face':
    upscale = 1
    align_method = 'nofix'

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTHONUNBUFFERED'] = '1'
gc.collect()
torch.cuda.empty_cache()
print(f'Free RAM: {psutil.virtual_memory().available/1024**3:.2f} GiB')

cmd = [
    'python', '-u', 'test_osediff.py',
    '-i', upload_folder,
    '-o', result_folder,
    '--upscale', str(upscale),
    '--osediff_path', osediff_path,
    '--pretrained_model_name_or_path', 'preset/models/stable-diffusion-2-1-base',
    '--ram_path', 'preset/models/ram_swin_large_14m.pth',
    '--ram_ft_path', 'preset/models/DAPE.pth',
    '--align_method', align_method,
    '--mixed_precision', mixed_precision,
]
if merge_and_unload_lora:
    cmd += ['--merge_and_unload_lora', 'True']
if low_vram_tiled_sampling:
    cmd += [
        '--vae_encoder_tiled_size', str(vae_encoder_tiled_size),
        '--vae_decoder_tiled_size', str(vae_decoder_tiled_size),
        '--latent_tiled_size', str(latent_tiled_size),
        '--latent_tiled_overlap', str(latent_tiled_overlap),
    ]

print('Command:', ' '.join(cmd))

# --- Run with full log capture ------------------------------------------------
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            bufsize=1, universal_newlines=True)
captured_lines = []
for line in process.stdout:
    captured_lines.append(line)
    if show_log:
        print(line, end='')
process.wait()

full_log = ''.join(captured_lines)

# --- Verify the actual result, not just the return code ----------------------
result_files_after = []
if os.path.isdir(result_folder):
    result_files_after = [f for f in os.listdir(result_folder) if os.path.isfile(os.path.join(result_folder, f))]

m_count = re.search(r'There are (\d+) images', full_log)
reported_count = int(m_count.group(1)) if m_count else None

if process.returncode != 0:
    print('\n⚠️ test_osediff.py exited with code', process.returncode)
    print('--- Last lines of the log (for diagnostics) ---')
    for line in captured_lines[-40:]:
        print(line, end='')
elif reported_count == 0 or not result_files_after:
    print('\n❌ The script exited without an error, BUT no images were processed or saved.')
    print(f'   .png input files found: {len(png_files_now)}')
    print(f'   Script reported "There are {reported_count} images."')
    print(f'   Files in result folder: {len(result_files_after)}')
    print('   Check the log above — the failure likely occurred inside the per-file processing loop.')
    raise RuntimeError('Processing failed: 0 images were produced.')
else:
    print(f'\n✅ Done! Processed images -> {result_folder}')
    print(f'   Script reported "There are {reported_count} images."')
    print(f'   Files in result folder: {len(result_files_after)}')
    for f in result_files_after:
        print(f'     -> {f}')

elapsed_time = time.time() - start_time_main
print(f'⏱️ Total execution time: {elapsed_time:.1f}s')
print(f'🖥️ RAM available after run: {psutil.virtual_memory().available/1024**3:.2f} GiB')

In [ ]:
#@title ##**Visualization** { display-mode: "form" }
import os
import base64
from io import BytesIO
import PIL.Image
from IPython.display import HTML, display

def is_image_file(filename):
    image_extensions = {'.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff'}
    return os.path.splitext(filename.lower())[1] in image_extensions

def image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

# test_osediff.py saves results directly to result_folder (without nested sampleNN/,
# like some other SR repos), but we still search for images recursively just in case.
def find_images(root):
    found = []
    for dirpath, _, fnames in os.walk(root):
        for fname in sorted(fnames):
            if is_image_file(fname):
                found.append(os.path.join(dirpath, fname))
    return sorted(found)

filenames_input_all = [f for f in os.listdir(upload_folder) if is_image_file(f)]
paths_output_all = find_images(result_folder)


input_by_base = {}
for f in filenames_input_all:
    base, ext = os.path.splitext(f)
    base_lower = base.lower()
    ext_lower = ext.lower()
    if base_lower not in input_by_base:
        input_by_base[base_lower] = f
    elif ext_lower == '.png':
        input_by_base[base_lower] = f

matched_pairs = []
for path_out in paths_output_all:
    out_filename = os.path.basename(path_out)
    out_base, _ = os.path.splitext(out_filename)
    out_base_lower = out_base.lower()
    if out_base_lower in input_by_base:
        in_filename = input_by_base[out_base_lower]
        matched_pairs.append((in_filename, path_out))

if not matched_pairs:
    print(f'Error: no matching image pairs found between {upload_folder} and {result_folder}.')
else:
    for idx, (filename, path_output) in enumerate(matched_pairs):
        try:
            # Загружаем изображения в ПОЛНОМ разрешении — без предварительного уменьшения.
            # Уменьшать будем только визуально (через CSS/размер контейнера),
            # а в base64/canvas всегда идёт оригинальный апскейленный пиксель.
            image_before = PIL.Image.open(os.path.join(upload_folder, filename))
            image_after = PIL.Image.open(path_output)

            if image_before.mode != 'RGB':
                image_before = image_before.convert('RGB')
            if image_after.mode != 'RGB':
                image_after = image_after.convert('RGB')

            # Размер, в котором картинка будет ПОКАЗАНА на экране (не влияет на пиксельные данные)
            display_max_width = 1000
            full_w, full_h = image_after.size
            if full_w > display_max_width:
                display_width = display_max_width
                display_height = int(full_h * display_max_width / full_w)
            else:
                display_width = full_w
                display_height = full_h

            before_base64 = image_to_base64(image_before)
            after_base64 = image_to_base64(image_after)

            uid = f"cmp{idx}"
            base_name = os.path.splitext(filename)[0]

            html_code = f"""
            <style>
                .download-btn-{uid} {{
                    background-color: #28a745;
                    color: white;
                    border: none;
                    border-radius: 4px;
                    padding: 5px 10px;
                    cursor: pointer;
                    display: flex;
                    align-items: center;
                    gap: 6px;
                    font-family: sans-serif;
                    font-size: 12px;
                    font-weight: bold;
                    flex-shrink: 0;
                    transition: background-color 0.15s ease-in-out;
                }}
                .download-btn-{uid}:hover {{
                    background-color: #218838;
                }}
                .download-btn-{uid}:active {{
                    background-color: #1e7e34;
                }}
            </style>
            <div style="width: {display_width}px; margin-bottom: 30px;">
                <div id="{uid}" style="position: relative; width: 100%; height: {display_height}px;">
                    <div style="position: relative; width: 100%; height: 100%; overflow: hidden;">
                        <img class="img-before" src="data:image/png;base64,{before_base64}" style="position: absolute; width: 100%; height: 100%;">
                        <div class="clip-wrap" style="position: absolute; width: 100%; height: 100%; overflow: hidden; clip-path: inset(0 0 0 50%);">
                            <canvas class="canvas-after" style="position: absolute; width: 100%; height: 100%; opacity: 1;"></canvas>
                        </div>
                    </div>
                    <div class="slider" style="position: absolute; top: 0; bottom: 0; width: 4px; background: white; cursor: ew-resize; left: 50%; box-shadow: 0 0 5px rgba(0,0,0,0.5);">
                        <div style="position: absolute; top: 50%; transform: translateY(-50%); width: 20px; height: 20px; background: white; border-radius: 50%; left: -8px;"></div>
                    </div>
                    <div style="position: absolute; top: 8px; left: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">Before</div>
                    <div style="position: absolute; top: 8px; right: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">After</div>
                </div>

                <div style="display: flex; flex-direction: column; gap: 8px; margin-top: 8px; font-family: sans-serif; font-size: 13px; color: inherit; min-width: 360px;">
                    <!-- Ряд 1: Смешивание оригинала -->
                    <div style="display: flex; align-items: center; gap: 10px;">
                        <span style="white-space: nowrap; flex-shrink: 0; min-width: 90px;">Blend original:</span>
                        <input type="range" class="blend-slider" min="0" max="100" value="100" step="1" style="flex: 1;">
                        <span class="blend-value" style="min-width: 45px; text-align: right; flex-shrink: 0;">100%</span>
                        <div style="width: 100px; flex-shrink: 0;"></div> <!-- Выравнивание кнопки -->
                    </div>

                    <!-- Ряд 2: Резкость и кнопка скачивания -->
                    <div style="display: flex; align-items: center; gap: 10px;">
                        <span style="white-space: nowrap; flex-shrink: 0; min-width: 90px;">Sharpness:</span>
                        <input type="range" class="sharpness-slider" min="0.0" max="7.0" value="0.0" step="0.1" style="flex: 1;">
                        <span class="sharpness-value" style="min-width: 45px; text-align: right; flex-shrink: 0;">0.0</span>
                        <button class="download-btn-{uid}" title="Download blended and sharpened image" style="width: 100px;">
                            <svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" style="display: inline-block;">
                                <path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"></path>
                                <polyline points="7 10 12 15 17 10"></polyline>
                                <line x1="12" y1="15" x2="12" y2="3"></line>
                            </svg>
                            Download
                        </button>
                    </div>
                </div>
            </div>
            <script>
                (function() {{
                    const root = document.getElementById('{uid}');
                    const slider = root.querySelector('.slider');
                    const container = root.querySelector('div:nth-child(1)');
                    const clipDiv = root.querySelector('.clip-wrap');
                    let isDragging = false;

                    slider.addEventListener('mousedown', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('mouseup', () => {{ isDragging = false; }});
                    document.addEventListener('mousemove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});

                    slider.addEventListener('touchstart', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('touchend', () => {{ isDragging = false; }});
                    document.addEventListener('touchmove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.touches[0].clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});
                }})();

                (function() {{
                    const root = document.getElementById('{uid}');
                    const wrapper = root.parentElement;
                    const blendSlider = wrapper.querySelector('.blend-slider');
                    const blendValue = wrapper.querySelector('.blend-value');
                    const sharpnessSlider = wrapper.querySelector('.sharpness-slider');
                    const sharpnessValue = wrapper.querySelector('.sharpness-value');

                    const canvasAfter = root.querySelector('.canvas-after');
                    const downloadBtn = wrapper.querySelector('.download-btn-{uid}');

                    // Загружаем изображения в JS (это уже полноразмерные данные)
                    const imgBeforeSource = new Image();
                    imgBeforeSource.src = "data:image/png;base64,{before_base64}";

                    const imgAfterSource = new Image();
                    imgAfterSource.src = "data:image/png;base64,{after_base64}";

                    let originalImgData = null;
                    let smoothedPixels = null;
                    let ctx = null;

                    // Алгоритм сглаживания SMOOTH (ядро 3х3: сумма коэффициентов 13, делитель 13)
                    function getSmoothedPixels(imgData) {{
                        const w = imgData.width;
                        const h = imgData.height;
                        const src = imgData.data;
                        const dst = new Uint8ClampedArray(src.length);
                        dst.set(src); // копируем каналы и границы

                        const kernel = [
                            1, 1, 1,
                            1, 5, 1,
                            1, 1, 1
                        ];
                        const divisor = 13;

                        for (let y = 1; y < h - 1; y++) {{
                            for (let x = 1; x < w - 1; x++) {{
                                let r = 0, g = 0, b = 0;
                                for (let ky = -1; ky <= 1; ky++) {{
                                    const rowOffset = (y + ky) * w * 4;
                                    for (let kx = -1; kx <= 1; kx++) {{
                                        const idx = rowOffset + (x + kx) * 4;
                                        const weight = kernel[(ky + 1) * 3 + (kx + 1)];
                                        r += src[idx] * weight;
                                        g += src[idx + 1] * weight;
                                        b += src[idx + 2] * weight;
                                    }}
                                }}
                                const dstIdx = (y * w + x) * 4;
                                dst[dstIdx]     = Math.min(255, Math.max(0, r / divisor));
                                dst[dstIdx + 1] = Math.min(255, Math.max(0, g / divisor));
                                dst[dstIdx + 2] = Math.min(255, Math.max(0, b / divisor));
                            }}
                        }}
                        return dst;
                    }}

                    // Алгоритм изменения резкости: экстраполяция оригинального и сглаженного массивов пикселей
                    function applySharpness(factor) {{
                        if (!originalImgData || !smoothedPixels) return;

                        const w = canvasAfter.width;
                        const h = canvasAfter.height;

                        if (factor === 1.0) {{
                            ctx.putImageData(originalImgData, 0, 0);
                            return;
                        }}

                        const src = originalImgData.data;
                        const smoothed = smoothedPixels;

                        const outputData = ctx.createImageData(w, h);
                        const dst = outputData.data;

                        for (let i = 0; i < src.length; i += 4) {{
                            // out = smoothed * (1 - factor) + original * factor
                            dst[i]     = Math.min(255, Math.max(0, smoothed[i] * (1 - factor) + src[i] * factor));
                            dst[i + 1] = Math.min(255, Math.max(0, smoothed[i + 1] * (1 - factor) + src[i + 1] * factor));
                            dst[i + 2] = Math.min(255, Math.max(0, smoothed[i + 2] * (1 - factor) + src[i + 2] * factor));
                            dst[i + 3] = src[i + 3]; // альфа-канал
                        }}
                        ctx.putImageData(outputData, 0, 0);
                    }}

                    imgAfterSource.onload = () => {{
                        // Canvas backing store = ПОЛНОЕ разрешение картинки After.
                        // Отображается он через CSS (width/height:100% контейнера),
                        // так что на экране может быть меньше, а в данных — полный размер.
                        canvasAfter.width = imgAfterSource.width;
                        canvasAfter.height = imgAfterSource.height;
                        ctx = canvasAfter.getContext('2d');

                        // Получаем пиксели оригинального "After"
                        ctx.drawImage(imgAfterSource, 0, 0);
                        originalImgData = ctx.getImageData(0, 0, canvasAfter.width, canvasAfter.height);

                        // Вычисляем размытие SMOOTH
                        smoothedPixels = getSmoothedPixels(originalImgData);

                        // Первичная отрисовка с фактором резкости 0.0
                        applySharpness(0.0);
                    }};

                    // Обработка слайдера резкости
                    sharpnessSlider.addEventListener('input', (e) => {{
                        const val = parseFloat(e.target.value);
                        sharpnessValue.textContent = val.toFixed(1);
                        applySharpness(val);
                    }});

                    // Обработка слайдера смешивания
                    blendSlider.addEventListener('input', (e) => {{
                        const val = e.target.value;
                        blendValue.textContent = val + '%';
                        canvasAfter.style.opacity = val / 100;
                    }});

                    // Логика скачивания: совмещает "Before", текущие настройки резкости "After" и прозрачность
                    // Итоговый canvas создаётся в ПОЛНОМ разрешении After (canvasAfter.width/height),
                    // "Before" растягивается drawImage'ом до тех же размеров — качество не режется.
                    downloadBtn.addEventListener('click', () => {{
                        const canvas = document.createElement('canvas');
                        const w = canvasAfter.width;
                        const h = canvasAfter.height;
                        canvas.width = w;
                        canvas.height = h;

                        const downloadCtx = canvas.getContext('2d');

                        // 1. Рисуем изображение "Before"
                        downloadCtx.drawImage(imgBeforeSource, 0, 0, w, h);

                        // 2. Накладываем "After" с учетом резкости и прозрачности
                        const opacity = parseFloat(blendSlider.value) / 100;
                        downloadCtx.globalAlpha = opacity;
                        downloadCtx.drawImage(canvasAfter, 0, 0, w, h);

                        // 3. Вызов скачивания файла в PNG
                        const link = document.createElement('a');
                        link.download = 'blended_{base_name}.png';
                        link.href = canvas.toDataURL('image/png');
                        link.click();
                    }});
                }})();
            </script>
            """

            display(HTML(html_code))

        except Exception as e:
            print(f'Error processing {filename} and {path_output}: {e}')

In [ ]:
#@title ##**Download Results** { display-mode: "form" }
import os
import zipfile
from google.colab import files

# Check if result_folder is defined
if 'result_folder' not in globals():
    result_folder = './results/user_upload'

# Define supported image extensions to exclude text logs, configs, etc.
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tiff', '.gif'}

if not os.path.isdir(result_folder):
    print(f"⚠️ Result folder not found: {result_folder}")
else:
    # Collect only image files in the result folder
    output_files = []
    for root, _, filenames in os.walk(result_folder):
        for filename in filenames:
            ext = os.path.splitext(filename.lower())[1]
            if ext in IMAGE_EXTENSIONS and not filename.startswith('.'):
                output_files.append(os.path.join(root, filename))

    output_files.sort()

    if len(output_files) == 0:
        print("⚠️ No output image files found to download.")
    elif len(output_files) == 1:
        # Download the single image directly without archiving
        target_file = output_files[0]
        print(f"Downloading file: {os.path.basename(target_file)}")
        files.download(target_file)
    else:
        # Archive multiple image files into a ZIP and download it
        zip_filename = 'OSEDiff_results.zip'
        if os.path.exists(zip_filename):
            os.remove(zip_filename)

        print(f"Archiving {len(output_files)} image files into {zip_filename}...")
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for file_path in output_files:
                # arcname=os.path.basename(file_path) keeps files at the archive root
                zipf.write(file_path, arcname=os.path.basename(file_path))

        print(f"Downloading archive: {zip_filename}")
        files.download(zip_filename)